In [8]:
import os
import chromadb
#import requests
from openai import OpenAI
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
from PyPDF2 import PdfReader
#from google.colab import userdata
#from google import genai
from sentence_transformers import CrossEncoder
from dotenv import load_dotenv

# Knowledge Base:

### Loading previous knowledge base:
-if loading previous knowledge base, just run the following cell   
-if building knowledge base, run all cells in this section

In [4]:
# chroma_client = chromadb.PersistentClient(path='/content/')
# collection = client.create_collection(
#     name="my_collection",
#     embedding_function=OpenAIEmbeddingFunction(
#         model_name="text-embedding-3-small"
#         api_key_env_var=OPENAI_API_KEY
#     )
# )

chroma_client = chromadb.PersistentClient(path='./chroma_db')
collection = chroma_client.get_or_create_collection(name="test_collection")

In [ ]:
def get_text_txt_md(file_path: str) -> str:
  with open(file_path, "r", encoding="utf-8") as f:
    text = f.read()
  return text

def get_text_pdf(pdf_path: str) -> str:
    """Extract raw text from a PDF file."""
    try:
        reader = PdfReader(pdf_path)
        return " ".join(page.extract_text() for page in reader.pages if page.extract_text())
    except Exception as e:
        raise RuntimeError(f"Error reading PDF: {e}")

In [ ]:
def chunk_text(text: str, chunk_size: int = 200, overlap: int = 25) -> list[str]:
    """Split text into manageable chunks for embeddings."""
    words = text.split()
    return [" ".join(words[i-overlap:i+chunk_size]) for i in range(overlap, len(words), chunk_size - overlap)]

In [ ]:
files_paths = ['toy_rag_data/Snake_wikiWikipedia.pdf', 'toy_rag_data/cool_math.pdf', 'toy_rag_data/its_nice_that.txt']
chunks = []
text = ""
for file in files_paths:
  if "pdf" in file:
    text = get_text_pdf(file)
  else:
    text = get_text_txt_md(file)
  chunks += chunk_text(text)

collection.upsert(
    documents=chunks,
    ids=[f"id{i}" for i in range(len(chunks))]
)

In [ ]:
#test retrieve:
collection.query(
      query_texts=["Who was the original creator?"],
      n_results=4
  )

# Generation:

In [ ]:
#loaded models
load_dotenv(dotenv_path="credentails.env")
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)
llm_model = "gpt-5.6-luna"
rr_model = CrossEncoder("cross-encoder/ettin-reranker-68m-v1")
query = "When was the snake game made?"

In [ ]:
def call_llm(model: str, prompt: str, client):
  try:
    response = client.responses.create(
      model=model,
      input=prompt,
    )
    return response.output_text
  except Exception as e:
      raise RuntimeError(f"LLM query failed: {e}")

In [ ]:
def filter_chunks(chunks, query, rr_model, max, threshold=0.0):
    """Rerank and filter chunks based on relevance to the query."""
    reranked = rr_model.rank(query=query, documents=chunks, return_documents=False, top_k=max)
    reranked = [item for item in reranked if item['score'] >= threshold]
    return reranked

In [ ]:
def source_list(reranked, res_k):
    """Create a list of dictionaries with source and content for each reranked chunk."""
    return [
        {
            'source': res_k['metadatas'][0][item['corpus_id']],
            'content': res_k['documents'][0][item['corpus_id']]
        }
        for item in reranked
    ]

In [ ]:
def gen_response(query, model, client, rr_model, collection, k, kp):
  res_k = collection.query(
      query_texts=[query],
      n_results=k
  )
  #print(res_k['metadatas'])
  
  reranked = filter_chunks(res_k['documents'][0], query, rr_model, threshold=0.5, max=kp)
  kp_chunks = [res_k['documents'][0][item['corpus_id']] for item in reranked]
  context = "\n".join(kp_chunks)
  prompt = f"Instruction: \nOnly use the following context to answer the question. If there is no context given, do not answer the question. If the context does not contain the answer, do not answer the question. \nQuestion: {query} \nContext: \n{context}"
  return call_llm(model, prompt, client)

In [ ]:
gen_response(query, llm_model, client, rr_model, collection, 10, 3)